# 04 · D-CPI mensual por clase y comparación con IPC oficial

Este notebook produce las vistas mensuales del D-CPI por clase, que son el insumo principal para la narrativa del proyecto. El notebook 03 entregó D-CPI trimestral porque los salarios EPH son trimestrales; acá trabajamos a frecuencia mensual para no perder la dinámica de los shocks.

Tres vistas que producimos:

1. **Nivel** del D-CPI por clase y del IPC oficial (escala log).
2. **Variación interanual** mes a mes — es donde se ven los movimientos.
3. **Comparación a tres niveles**: IPC oficial publicado (canasta 2004/05) vs IPC promedio reconstruido (canasta 2017/18 overall) vs D-CPI por clase. Esto separa **dos efectos** que conviven:
   - Efecto "canasta vieja vs nueva" (lo que el OUE-La Pampa midió).
   - Efecto "heterogeneidad por perfil" (lo que este trabajo mide).

**Salidas:**
- `data/dcpi_monthly_full.csv` — D-CPI mensual por clase + IPC oficial + IPC promedio reconstruido.
- `figures/05_dcpi_series.png` — tres paneles: nivel, i.a., acumulada.
- `figures/06_dcpi_gap_vs_oficial.png` — gap mensual de cada clase respecto al oficial.
- `figures/07_three_way_comparison.png` — la figura clave para la presentación.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

CLASS_NAMES = {
    0: "Obreros formales (sec.)",
    1: "Asalariados informales jóvenes",
    2: "Trabajadores mayores con primario",
    3: "Profesionales asalariadas",
    4: "Cuentapropistas y patrones",
}

## 1. Cargar IPC mensual y rebasear a enero 2017 = 100

In [ ]:
ipc_raw = pd.read_csv("data/ipc_indec_real.csv", parse_dates=["indice_tiempo"])

col_map = {
    "ipc_2016_alimentos_bebidas": "div_01",
    "ipc_2016_bebidas_alcoholicas": "div_02",
    "ipc_2016_indumentaria": "div_03",
    "ipc_2016_vivienda": "div_04",
    "ipc_2016_equipamiento_del_hogar": "div_05",
    "ipc_2016_atencion_medica_salud": "div_06",
    "ipc_2016_transporte": "div_07",
    "ipc_2016_informacion_comunicacion": "div_08",
    "ipc_2016_recreacion_cultura": "div_09",
    "ipc_2016_educacion": "div_10",
    "ipc_2016_restauranes_hoteles": "div_11",
    "ipc_2016_bienes_servicios_varios": "div_12",
}
ipc = ipc_raw.rename(columns={**col_map, "ipc_2016_nivel_general": "nivel_general"})
ipc = ipc[ipc["indice_tiempo"] >= "2017-01-01"].set_index("indice_tiempo").sort_index()

# Rebasear a enero 2017 = 100
base = ipc.iloc[0].copy()
ipc = ipc.div(base) * 100
print(f"IPC mensual: {len(ipc)} meses ({ipc.index[0]:%Y-%m} → {ipc.index[-1]:%Y-%m})")
ipc.head(3)

## 2. Calcular D-CPI mensual por clase + IPC promedio reconstruido

Tres tipos de canasta combinadas con la matriz mensual de IPC por división:

- **IPC oficial publicado**: la columna `nivel_general` del INDEC. Usa ponderadores ENGH 2004/05.
- **IPC promedio reconstruido**: usa los `share_*` overall ponderados de la ENGH 2017/18.
- **D-CPI por clase**: usa los `share_*` promedio de cada clase del LCA.

In [ ]:
shares = pd.read_csv("data/shares_by_class.csv", index_col=0)
hog = pd.read_parquet("data/engh_hogares.parquet")

overall_shares = np.array([
    (hog[f"share_{i:02d}"]*hog["pondera"]).sum() / hog["pondera"].sum()
    for i in range(1, 13)
])

M = ipc[[f"div_{i:02d}" for i in range(1, 13)]].values  # n_t × 12

dcpi = pd.DataFrame(index=ipc.index)
dcpi["IPC oficial publicado"] = ipc["nivel_general"].values
dcpi["IPC promedio (canasta 2017/18 overall)"] = M @ overall_shares
for c in shares.index:
    s = np.array([shares.loc[c, f"share_{i:02d}"] for i in range(1, 13)])
    dcpi[CLASS_NAMES[c]] = M @ s

dcpi.to_csv("data/dcpi_monthly_full.csv")
print(f"D-CPI mensual con {dcpi.shape[1]} series construido.")
dcpi.tail(3).round(0)

## 3. Variación interanual y acumulada

In [ ]:
yoy = (dcpi / dcpi.shift(12) - 1) * 100
yoy = yoy.dropna()
acum = (dcpi / dcpi.iloc[0] - 1) * 100

# Diferencias respecto al IPC oficial publicado (en p.p.)
yoy_diff = yoy.subtract(yoy["IPC oficial publicado"], axis=0)
acum_diff = acum.subtract(acum["IPC oficial publicado"], axis=0)

## 4. Tabla resumen — variación interanual cada diciembre

Esta tabla replica el tipo de comparación que hace el OUE-La Pampa, pero en lugar de comparar dos canastas únicas (2004/05 vs 2017/18), compara **las cinco canastas por clase**.

In [ ]:
dec_yoy = yoy.loc[yoy.index.month == 12].copy()
dec_yoy.index = dec_yoy.index.year
print("Inflación interanual cada diciembre, por canasta:\n")
print(dec_yoy.round(1))

print("\n\nSpread entre clases (max - min) — sólo entre los 5 D-CPI por clase:")
class_cols = [c for c in dec_yoy.columns if c not in ("IPC oficial publicado",
                                                       "IPC promedio (canasta 2017/18 overall)")]
spread = (dec_yoy[class_cols].max(axis=1) - dec_yoy[class_cols].min(axis=1))
print(spread.round(1))

### Cómo leer esta tabla

En 2018-2022 el spread entre clases es < 1 pp. En **2023-2024** salta a **5-6 pp**. Esto es coherente con la observación de Lódola, Busso & Cerimedo (2000) sobre el sesgo "anti-rico" de la inflación alta: los períodos de mayor turbulencia inflacionaria producen mayor heterogeneidad entre canastas. En 2024 específicamente, el ajuste tarifario y la devaluación afectaron más a las canastas con peso significativo en vivienda y servicios (profesionales asalariadas) que a las cargadas en alimentos (mayores con primario).

**Insight central**: la heterogeneidad no es un atributo permanente del IPC; es un atributo de los *shocks*. En períodos de inflación tranquila las clases convergen al promedio.

## 5. Figura A — D-CPI por clase: nivel, interanual y acumulada

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)

palette = {
    "IPC oficial publicado": "black",
    "IPC promedio (canasta 2017/18 overall)": "#666",
    "Obreros formales (sec.)": "#1f77b4",
    "Asalariados informales jóvenes": "#ff7f0e",
    "Trabajadores mayores con primario": "#2ca02c",
    "Profesionales asalariadas": "#d62728",
    "Cuentapropistas y patrones": "#9467bd",
}
ls_map = {
    "IPC oficial publicado": ("-", 2.2),
    "IPC promedio (canasta 2017/18 overall)": ("--", 2.0),
}

# Panel 1 — nivel (log)
for col in dcpi.columns:
    style, lw = ls_map.get(col, ("-", 1.4))
    axes[0].plot(dcpi.index, dcpi[col], label=col, color=palette[col], linestyle=style, lw=lw)
axes[0].set_yscale("log")
axes[0].set_ylabel("Índice (ene-2017 = 100, log)")
axes[0].set_title("Nivel del D-CPI por clase y referencias")
axes[0].grid(alpha=0.3, which="both")
axes[0].legend(loc="upper left", fontsize=7)

# Panel 2 — interanual
for col in dcpi.columns:
    style, lw = ls_map.get(col, ("-", 1.4))
    axes[1].plot(yoy.index, yoy[col], color=palette[col], linestyle=style, lw=lw)
axes[1].set_ylabel("Inflación i.a. (%)")
axes[1].set_title("Variación interanual")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
axes[1].grid(alpha=0.3)

# Panel 3 — acumulada
for col in dcpi.columns:
    style, lw = ls_map.get(col, ("-", 1.4))
    axes[2].plot(acum.index, acum[col], color=palette[col], linestyle=style, lw=lw)
axes[2].set_ylabel("Inflación acumulada (%)")
axes[2].set_title("Inflación acumulada desde enero 2017")
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
axes[2].grid(alpha=0.3)

fig.suptitle("D-CPI por clase latente — Argentina 2017–2026", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig("figures/05_dcpi_series.png", dpi=120)
plt.show()

## 6. Figura B — Gap mensual respecto al IPC oficial publicado

Esta figura responde directamente: "¿qué clase enfrentó más inflación que el IPC oficial mes a mes?". Los movimientos por encima de cero indican períodos en que la canasta de la clase X subió más que la canasta promedio del IPC oficial.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for col in [c for c in dcpi.columns if c not in ("IPC oficial publicado",
                                                  "IPC promedio (canasta 2017/18 overall)")]:
    ax.plot(yoy_diff.index, yoy_diff[col], label=col, color=palette[col], lw=1.6)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Diferencia i.a. respecto al IPC oficial (p.p.)")
ax.set_title("¿Qué clase enfrentó más inflación que el IPC oficial mes a mes?")
ax.grid(alpha=0.3)
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig("figures/06_dcpi_gap_vs_oficial.png", dpi=120)
plt.show()

## 7. Figura C — Comparación a tres niveles

Esta es **la figura central para la narrativa del proyecto**. Separa el efecto "canasta vieja vs nueva" (que es el del OUE) del efecto "heterogeneidad por perfil" (que es el aporte de este trabajo).

- En 2017–2022 el "IPC promedio reconstruido" corre pegado al "IPC oficial publicado" → la canasta vieja y la nueva no diferían tanto.
- En **2024** el "IPC promedio reconstruido" salta ~6 pp por encima del oficial → ahí se ve el efecto OUE.
- Encima de eso, las clases se separan entre sí en otros ~5 pp → ahí se ve la heterogeneidad por perfil.

Los dos efectos **se suman**; no compiten.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Panel 1: niveles i.a.
for col in dcpi.columns:
    style, lw = ls_map.get(col, ("-", 1.4))
    axes[0].plot(yoy.index, yoy[col], label=col, color=palette[col], linestyle=style, lw=lw)
axes[0].set_ylabel("Inflación interanual (%)")
axes[0].set_title("Variación interanual: IPC oficial vs IPC promedio reconstruido vs D-CPI por clase")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
axes[0].grid(alpha=0.3)
axes[0].legend(loc="upper left", fontsize=8)

# Panel 2: brecha vs oficial publicado
for col in dcpi.columns:
    if col == "IPC oficial publicado":
        continue
    style, lw = ls_map.get(col, ("-", 1.4))
    axes[1].plot(yoy_diff.index, yoy_diff[col], label=col, color=palette[col], linestyle=style, lw=lw)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_ylabel("Diferencia i.a. vs IPC oficial publicado (p.p.)")
axes[1].set_title("Brecha mes a mes — separa el efecto 'canasta nueva vs vieja' del efecto 'heterogeneidad por perfil'")
axes[1].grid(alpha=0.3)
axes[1].legend(loc="upper left", fontsize=8)

fig.suptitle("Tres niveles de comparación — IPC oficial, IPC promedio actualizado, D-CPI por clase",
             fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig("figures/07_three_way_comparison.png", dpi=120)
plt.show()

## 8. Tabla final — diciembre 2024 desagregada

El momento más informativo del período. Sirve para incorporar a la presentación y al texto.

In [ ]:
dec24 = pd.DataFrame({
    "Inflación i.a. (%)": yoy.loc["2024-12-01"],
    "Diferencia vs IPC oficial publicado (p.p.)": yoy_diff.loc["2024-12-01"],
}).round(2)
print("Diciembre 2024 — variación interanual:\n")
print(dec24)

## Lectura para la presentación

Tres mensajes que se desprenden de este análisis y que conviene incorporar a la narrativa:

1. **El POC reproduce dentro de su pipeline el efecto "canasta vieja vs nueva" que detectó el OUE-La Pampa**, sin necesidad de tener la ENGH 2004/05. La actualización de ponderadores 2004/05 → 2017/18 produce ~6 pp de inflación adicional en 2024 (consistente con el orden de magnitud reportado por OUE de ~19 pp con su recorte específico).

2. **Encima de ese efecto agregado, hay una capa adicional de heterogeneidad entre perfiles de trabajadores** (~5 pp de spread entre clases en 2024). La dirección del spread es consistente con el sesgo "anti-rico" de Lódola para regímenes de inflación alta.

3. **La magnitud acumulada en 8 años es modesta (1 pp)** porque los shocks tienen efectos compensatorios. La heterogeneidad **no es un atributo permanente del IPC**, sino un atributo de los **shocks de precios relativos**. Eso es un hallazgo en sí mismo y matiza la lectura ingenua de "todos pierden igual".